# GRAFICO 2D Y 3D DEL PATRON POLAR

In [ ]:
%load_ext autoreload
%autoreload 2
from mic_array.patron import MicArray
from filterbank import FilterBank

# CALIBRACIÓN

In [ ]:
ma = MicArray.from_tensor('data/tensores/forte_full_aligned.npz')

In [ ]:
ma.calibrate(
    path="data/media",
    array_pattern="mic_{MIC}_ang_cal.wav",
    ref_pattern="mic_ref_ang_cal.wav"
)

ma.to_spl()   # tensor pasa a Pa

# DETECCIÓN DE NOTAS

- La detección se hace a partir de analizar el micrófono de referencia.
- Se crea una lista llamada segments donde se guarda el inicio y fin de cada intervalo para cada giro en azimuth.

In [ ]:
ma.scale = {'Fa4':349.23,'Sol4':392.,'La4':440.,'Sib4':466.16,
            'Do5':523.25,'Re5':587.33,'Mi5':659.25,'Fa5':698.46}

# segments = ma.detect_notes(start_s=0.3)

# Si hay notas que no detecta, bajar confidence
# Más conservador — incluye más frames de transición
segments = ma.detect_notes(start_s=0.3, confidence=0.2, gradient_thresh=50.0)


## Calibración manual del ajuste para notas que no se detectaron bien

In [ ]:
# 3a. Corregir un segmento mal ubicado
ma.edit_segment(segments, azimuth=0, note='La4', start_s=1.86, end_s=2.39)
ma.edit_segment(segments, azimuth=0, note='Sib4', start_s=2.42, end_s=2.96)
ma.edit_segment(segments, azimuth=0, note='Mi5', start_s=4.20, end_s=4.70)
ma.edit_segment(segments, azimuth=0, note='Fa5', start_s=4.78, end_s=5.50)

ma.edit_segment(segments, azimuth=10, note='La4', start_s=1.97, end_s=2.51)
ma.edit_segment(segments, azimuth=10, note='Mi5', start_s=4.50, end_s=4.89)
ma.edit_segment(segments, azimuth=10, note='Fa5', start_s=5, end_s=5.70)

ma.edit_segment(segments, azimuth=40, note='Fa5', start_s=4.80, end_s=5.5)
ma.edit_segment(segments, azimuth=50, note='Fa5', start_s=4.73, end_s=5.5)

ma.edit_segment(segments, azimuth=70, note='Mi5', start_s=4.50, end_s=4.95)

ma.edit_segment(segments, azimuth=90, note='Mi5', start_s=4.95, end_s=5.53)

ma.edit_segment(segments, azimuth=100, note='Sol4', start_s=1.30, end_s=1.85)
ma.edit_segment(segments, azimuth=100, note='Sib4', start_s=2.55, end_s=3.1)
ma.edit_segment(segments, azimuth=100, note='Re5', start_s=3.83, end_s=4.35)
ma.edit_segment(segments, azimuth=100, note='Fa5', start_s=5, end_s=5.80)

ma.edit_segment(segments, azimuth=110, note='Mi5', start_s=4.38, end_s=4.82)
ma.edit_segment(segments, azimuth=110, note='Fa5', start_s=4.90, end_s=5.60)

ma.edit_segment(segments, azimuth=120, note='La4', start_s=1.93, end_s=2.5)
ma.edit_segment(segments, azimuth=120, note='Mi5', start_s=4.38, end_s=4.93)
ma.edit_segment(segments, azimuth=120, note='Fa5', start_s=5, end_s=5.8)

ma.edit_segment(segments, azimuth=130, note='Fa4', start_s=0.62, end_s=1.09)
ma.edit_segment(segments, azimuth=130, note='Re5', start_s=3.59, end_s=4.08)
ma.edit_segment(segments, azimuth=130, note='Mi5', start_s=4.15, end_s=4.63)
ma.edit_segment(segments, azimuth=130, note='Fa5', start_s=4.73, end_s=5.45)


ma.edit_segment(segments, azimuth=150, note='Do5', start_s=3.29, end_s=3.79)
ma.edit_segment(segments, azimuth=150, note='Re5', start_s=3.87, end_s=4.37)
ma.edit_segment(segments, azimuth=150, note='Mi5', start_s=4.49, end_s=5.02)
ma.edit_segment(segments, azimuth=150, note='Fa5', start_s=5.08, end_s=5.88)

ma.edit_segment(segments, azimuth=160, note='La4', start_s=1.95, end_s=2.51)
ma.edit_segment(segments, azimuth=160, note='Re5', start_s=3.85, end_s=4.37)
ma.edit_segment(segments, azimuth=160, note='Mi5', start_s=4.44, end_s=5.05)

ma.edit_segment(segments, azimuth=170, note='La4', start_s=1.97, end_s=2.51)
ma.edit_segment(segments, azimuth=170, note='Mi5', start_s=4.50, end_s=4.89)
ma.edit_segment(segments, azimuth=170, note='Fa5', start_s=5, end_s=5.70)


ma.edit_segment(segments, azimuth=180, note='La4', start_s=1.86, end_s=2.39)
ma.edit_segment(segments, azimuth=180, note='Sib4', start_s=2.42, end_s=2.96)
ma.edit_segment(segments, azimuth=180, note='Mi5', start_s=4.20, end_s=4.70)
ma.edit_segment(segments, azimuth=180, note='Fa5', start_s=4.78, end_s=5.50)

Se aplica la mascara al tensor para extraer las notas de cada mic.

In [ ]:
ma.extract_all_notes(segments)

Cada nota queda almacenada en un tensor similar al que tiene todos los audios organizado por azimuth, theta y samples.
Así al ser 8 notas se tienen 8 tensores que solo contienen audio de cada nota, sin silencios.

In [ ]:
ma.notes['Mi5'] # por cada nota se crea un nuevo tensor que solo contiene esa nota para todas las mediciones

Se puede escuchar una nota para saber si esta bien o no.

In [ ]:
ma.notes['Mi5'].listen(azimuth=100, theta=100)

# ANÁLISIS DE TODA LA SEñAL

## Cálculo de SPL por bandas

Al ya tener calibrados todos los micrófonos, toca calcular el SPL por bandas de 1/3 de octava.

`compute_spl()` divide el audio de cada micrófono en ventanas cortas (50 ms por defecto) y aplica un criterio de actividad de voz (VAD): descarta las ventanas cuyo nivel RMS esté por debajo de un umbral (`threshold_spl = 30 dB SPL`), eliminando silencios y pausas entre notas. Sobre los frames activos calcula el RMS energético por banda mediante FFT y lo expresa en dB SPL (ref. 20 µPa).

El resultado es una matriz `spl_levels` de forma `(azimuths × thetas × bandas)` con el nivel absoluto en cada posición del arreglo para cada banda de frecuencia.


In [ ]:
ma.compute_spl()

In [ ]:
ma.plot_spectrum()

# PATRON POLAR 2D

## Cálculo de directividad

`compute_directivity()` calcula el patrón polar de directividad de la cantante en bandas de 1/3 de octava. El proceso tiene dos pasos.

---

### El problema: la cantante no canta igual en cada toma

El experimento consiste en que la cantante canta la misma escala de notas mientras el arreglo de micrófonos la registra desde distintos ángulos de azimuth (0°, 10°, 20°…). El problema es que en cada toma ella puede haber emitido con un nivel diferente: tal vez en az=20° cantó el Sol4 a 80 dB SPL, y en az=100° lo cantó a 76 dB SPL. Si simplemente comparáramos los niveles de los micrófonos entre tomas, no sabríamos si la diferencia se debe a que el sonido radia distinto en esa dirección, o simplemente a que cantó más fuerte o más suave.

---

### Paso 1 — Cancelación por toma (micrófono de referencia)

Para resolver esto, en cada toma hay un micrófono de referencia posicionado a 1 metro de la boca, en el eje frontal. Como grabó simultáneamente a los demás micrófonos, capturó exactamente cuánto emitió la cantante en esa toma. Al restarle ese nivel a todos los demás micrófonos, la variabilidad de fonación queda cancelada:

$$\text{SPL}_{rel}[az,\ \theta,\ banda] = \text{SPL}[az,\ \theta,\ banda] - \text{SPL}[az,\ mic_{ref},\ banda]$$

**Ejemplo concreto con Sol4:**

| Toma | Ref capturó | mic_1 (θ=0°) capturó | Resultado (mic_1 − ref) |
|------|-------------|----------------------|--------------------------|
| az = 20°  | 80 dB SPL | 73 dB SPL | **−7 dB** |
| az = 100° | 76 dB SPL | 66 dB SPL | **−10 dB** |

Los −7 dB y −10 dB ya no dependen de cuánto cantó: reflejan solo la relación geométrica entre el micrófono y la boca en esa dirección. La diferencia de 3 dB entre ambos ángulos es información real del patrón de directividad.

---

### Paso 2 — Normalización al on-axis (θ=0°, az=0°)

Después de la cancelación, se elige una dirección como referencia absoluta del patrón: la dirección frontal a la altura de la boca (az=0°, θ=0°). Se resta ese valor a todo el patrón, de modo que esa dirección quede exactamente en 0 dB:

$$\text{SPL}_{norm}[az,\ \theta,\ banda] = \text{SPL}_{rel}[az,\ \theta,\ banda] - \text{SPL}_{rel}[az=0°,\ \theta=0°,\ banda]$$

El resultado es el **patrón de directividad relativo**: cuántos dB radia la fuente en cada dirección respecto al frente, para cada banda de frecuencia.

> Antes de normalizar, el nivel absoluto en dB SPL del on-axis se guarda en `dir_ref_spl`. Ese valor sirve para saber cuánta energía real hay en cada banda (sin que la normalización lo borre).

In [ ]:
ma.compute_directivity(bands='1/3', threshold_spl=30, ref_azimuth=0, ref_theta_plot=0)

In [ ]:
ma.listen_band(500)

In [ ]:
ma.plot_polar_2d(
    theta=0,
    source='directivity',
    freq=315,           # None = broadband
    db_range=[-30, 12],
    tick_step=6,
)

In [ ]:
ma.plot_polar_3d(freq=315, source='directivity', mirror=False, db_range=[-15, 3], interp_method='cubic', interp_deg=1)

In [ ]:
# ma.plot_directivity_sphere(freq=500, colorscale='plasma')
ma.plot_directivity_sphere(freq=630, colorscale='viridis', interp_deg = None)
# ma.plot_directivity_sphere(freq=500, colorscale='RdYlBu_r')
# ma.plot_directivity_sphere(freq=500, colorscale=[[0.0, 'blue'], [1.0, 'red']])

In [ ]:
ma.compute_directivity_notes(bands='1/3', threshold_spl=30, ref_azimuth=0, ref_theta_plot=0)

In [ ]:
ma.notes['Sol4'].plot_polar_2d(
    theta=0,
    source='directivity',
    freq=500,           # None = broadband
    db_range=[-30, 12],
    tick_step=6,
)

In [ ]:
# Y después por nota:
ma.notes['La4'].plot_polar_3d(freq=500, source='directivity', db_range=[-15, 3])

In [ ]:
ma.notes['Sol4'].plot_directivity_sphere(freq=630, colorscale='viridis',  interp_deg = None)

# TEST RUIDO

In [ ]:
raise SystemExit("— stop —")

In [ ]:
# ── 3. Cálculo de SPL activo (RMS sobre frames activos) ───────────
ma.compute_spl(bands='1/3', threshold_spl=30, window_ms=50)

In [ ]:
# ── 4. Normalización de directividad (ref = 0 dB) ─────────────────
ma.normalize(type='spl', ref_azimuth=0, ref_theta=0)

# Calculo de LEQ

Es medio al pedo porque hay silencios que tiran abajo el LEQ

## Cálculo del LEQ para todo el audio 

In [ ]:

# Para más control (fmin/fmax, orden, resoluciones 1/6, 1/12, 1/24) usá FilterBank directo:
# fb = FilterBank(sr=44100, bands='1/3', fmin=20, fmax=20000)
# freqs, levels = fb.leq(signal, method='iir')

# Desde MicArray — guarda los resultados en ma.leq_freqs y ma.leq_levelscoo

# Durante desarrollo — rápido
ma.compute_leq(bands='1/3', method='fft')

## Cálculo del LEQ para cada nota

Antes de ejecutar esto debe correrse ma.extract_all_notes(segments)

In [ ]:
ma.compute_leq_notes() 

# Cálculo SPL

El cálculo de SPL lo hace midiendo solo los intervalos donde hay señal activa, es decir sin silencios. Esto lo realiza tomando ventanas y tomando como valores válidos solo aquellos en los que se supera un umbral. 

In [ ]:
# Calcula SPL activo (broadband + por banda)
ma.compute_spl(bands='1/3', threshold_spl=30, window_ms=50)

In [ ]:
# Rango fijo 60–114 dB SPL, círculos cada 10 dB
ma.plot_polar_2d(theta=0, source='spl', vrange=[60, 90], tick_step=10)

# Normalización

Deben igualarse los niveles de todas las tomas ya que puede haber diferencias tanto de nivel como de calibración

## Para toda la señal

Se debe elegir respecto a que mic calibrar, esto se define con el theta. Si es con la calibración es theta='ref'. 
También se define respecto a que ángulo de azimuth o giro hacer la normalización 

In [ ]:
ma.normalize_leq(theta='ref', ref_azimuth=0)

In [ ]:


# Para las notas
ma.normalize_leq_notes(theta='ref', ref_azimuth=0)

In [ ]:
ma.plot_polar_2d(theta=0)

In [ ]:
ma.plot_polar_2d(theta=0, freq=315, dynamic_range=30, normalize=True)